# QuickPay FinTech Operations – Python Pipeline
**Covers:** Part 3 (Reconciliation Workflow) · Part 4 (JSON Normalization) · Dashboard Source Outputs

In [ ]:
import pandas as pd
import numpy as np
import json
import re
import os

RAW   = '../01_data/raw/'
OUT   = '../01_data/processed/'
os.makedirs(OUT, exist_ok=True)
print("Libraries loaded. Output directory ready.")

## Part 3 – Reconciliation Workflow
### 3.1 Load Files

In [ ]:
ledger  = pd.read_csv(RAW + 'ledger.csv')
gateway = pd.read_csv(RAW + 'gateway.csv')

print(f"Ledger : {ledger.shape[0]} rows × {ledger.shape[1]} cols")
print(f"Gateway: {gateway.shape[0]} rows × {gateway.shape[1]} cols")
print()
print("Ledger preview:")
display(ledger)
print("Gateway preview:")
display(gateway)

### 3.2 Validation – Duplicates & Nulls

In [ ]:
# Duplicates
l_dups = ledger.duplicated('transaction_id').sum()
g_dups = gateway.duplicated('transaction_id').sum()
print(f"Ledger duplicate transaction_ids : {l_dups}")
print(f"Gateway duplicate transaction_ids: {g_dups}")

# Nulls
print("\nLedger nulls:")
print(ledger.isnull().sum())
print("\nGateway nulls:")
print(gateway.isnull().sum())

### 3.3 Missing in Gateway (in ledger but not in gateway)

In [ ]:
ledger_ids  = set(ledger['transaction_id'])
gateway_ids = set(gateway['transaction_id'])

missing_in_gateway = ledger[~ledger['transaction_id'].isin(gateway_ids)].copy()
print(f"Records in ledger but MISSING in gateway: {len(missing_in_gateway)}")
display(missing_in_gateway)

missing_in_gateway.to_csv(OUT + 'missing_in_gateway.csv', index=False)
print("Saved: missing_in_gateway.csv")

### 3.4 Missing in Ledger (in gateway but not in ledger)

In [ ]:
missing_in_ledger = gateway[~gateway['transaction_id'].isin(ledger_ids)].copy()
print(f"Records in gateway but MISSING in ledger: {len(missing_in_ledger)}")
display(missing_in_ledger)

missing_in_ledger.to_csv(OUT + 'missing_in_ledger.csv', index=False)
print("Saved: missing_in_ledger.csv")

### 3.5 Amount Mismatches

In [ ]:
common = ledger.merge(gateway, on='transaction_id', suffixes=('_ledger','_gateway'))

amount_mismatches = common[
    common['amount_usd_ledger'] != common['amount_usd_gateway']
].copy()
amount_mismatches['amount_diff'] = (
    amount_mismatches['amount_usd_gateway'] - amount_mismatches['amount_usd_ledger']
)

print(f"Amount mismatches: {len(amount_mismatches)}")
display(amount_mismatches[['transaction_id','merchant_id_ledger',
                            'amount_usd_ledger','amount_usd_gateway','amount_diff']])

amount_mismatches.to_csv(OUT + 'amount_mismatches.csv', index=False)
print("Saved: amount_mismatches.csv")

### 3.6 Status Mismatches

In [ ]:
status_mismatches = common[
    common['status_ledger'] != common['status_gateway']
].copy()

print(f"Status mismatches: {len(status_mismatches)}")
display(status_mismatches[['transaction_id','merchant_id_ledger',
                            'status_ledger','status_gateway']])

status_mismatches.to_csv(OUT + 'status_mismatches.csv', index=False)
print("Saved: status_mismatches.csv")

### 3.7 Full Reconciliation Report

In [ ]:
all_ids = sorted(ledger_ids | gateway_ids)
recon_rows = []

for tid in all_ids:
    l_row = ledger[ledger['transaction_id'] == tid]
    g_row = gateway[gateway['transaction_id'] == tid]
    in_l, in_g = not l_row.empty, not g_row.empty

    if in_l and in_g:
        l, g = l_row.iloc[0], g_row.iloc[0]
        issues = []
        if l['amount_usd'] != g['amount_usd']: issues.append('amount_mismatch')
        if l['status']     != g['status']:      issues.append('status_mismatch')
        recon_rows.append({
            'transaction_id'         : tid,
            'in_ledger'              : True,
            'in_gateway'             : True,
            'ledger_amount'          : l['amount_usd'],
            'gateway_amount'         : g['amount_usd'],
            'ledger_status'          : l['status'],
            'gateway_status'         : g['status'],
            'reconciliation_status'  : 'matched' if not issues else '|'.join(issues)
        })
    elif in_l:
        l = l_row.iloc[0]
        recon_rows.append({
            'transaction_id'         : tid,
            'in_ledger'              : True,
            'in_gateway'             : False,
            'ledger_amount'          : l['amount_usd'],
            'gateway_amount'         : None,
            'ledger_status'          : l['status'],
            'gateway_status'         : None,
            'reconciliation_status'  : 'missing_in_gateway'
        })
    else:
        g = g_row.iloc[0]
        recon_rows.append({
            'transaction_id'         : tid,
            'in_ledger'              : False,
            'in_gateway'             : True,
            'ledger_amount'          : None,
            'gateway_amount'         : g['amount_usd'],
            'ledger_status'          : None,
            'gateway_status'         : g['status'],
            'reconciliation_status'  : 'missing_in_ledger'
        })

recon_df = pd.DataFrame(recon_rows)
print("Reconciliation Report:")
display(recon_df)
recon_df.to_csv(OUT + 'reconciliation_report.csv', index=False)
print("Saved: reconciliation_report.csv")

### 3.8 Summary Metrics

In [ ]:
risk_ids = set(missing_in_gateway['transaction_id'])
amt_risk = (
    missing_in_gateway['amount_usd'].sum()
    + amount_mismatches['amount_usd_ledger'].sum()
)

metrics = {
    "total_ledger_rows"          : int(len(ledger)),
    "total_gateway_rows"         : int(len(gateway)),
    "missing_in_gateway_count"   : int(len(missing_in_gateway)),
    "missing_in_ledger_count"    : int(len(missing_in_ledger)),
    "amount_mismatch_count"      : int(len(amount_mismatches)),
    "status_mismatch_count"      : int(len(status_mismatches)),
    "reconciliation_issue_count" : int(len(recon_df[recon_df['reconciliation_status']!='matched'])),
    "ledger_total_amount"        : round(float(ledger['amount_usd'].sum()), 2),
    "gateway_total_amount"       : round(float(gateway['amount_usd'].sum()), 2),
    "amount_at_risk"             : round(float(amt_risk), 2)
}

print(json.dumps(metrics, indent=2))

with open('../04_python/summary_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("Saved: summary_metrics.json")

---
## Part 4 – JSON Normalization
### 4.1 Load & Inspect

In [ ]:
with open(RAW + 'api_response_sample.json') as f:
    api_data = json.load(f)

print("Top-level keys:", list(api_data.keys()))
print("Source         :", api_data['source'])
print("Generated at   :", api_data['generated_at'])
print("Number of batches:", len(api_data['batches']))
for b in api_data['batches']:
    print(f"  Batch {b['batch_id']}: merchant={b['merchant']['merchant_name']}, "
          f"settlements={len(b['settlements'])}")

### 4.2 Flatten Nested Structure

In [ ]:
rows = []
for batch in api_data['batches']:
    m = batch['merchant']
    for s in batch['settlements']:
        rows.append({
            'batch_id'        : batch['batch_id'],
            'merchant_id'     : m['merchant_id'],
            'merchant_name'   : m['merchant_name'],
            'merchant_region' : m['region'],
            'settlement_id'   : s['settlement_id'],
            'amount_usd'      : s['amount_usd'],
            'status'          : s['status'],
            'processed_at'    : s['processed_at'],
            'bank_name'       : s['bank']['name'],
            'bank_country'    : s['bank']['country'],
            'generated_at'    : api_data['generated_at'],
            'source'          : api_data['source'],
        })

api_df = pd.DataFrame(rows)

# Convert datetime fields
api_df['processed_at'] = pd.to_datetime(api_df['processed_at'])
api_df['generated_at'] = pd.to_datetime(api_df['generated_at'])

print(f"Normalized: {api_df.shape[0]} rows × {api_df.shape[1]} columns")
print("\nColumn names:", api_df.columns.tolist())
print("\nData types:")
print(api_df.dtypes)
display(api_df)

### 4.3 Save Normalized Output

In [ ]:
api_df.to_csv(OUT + 'api_normalized.csv', index=False)
print("Saved: api_normalized.csv")
print("Preview:")
display(api_df.head())

---
## Dashboard Support Outputs
### Daily Summary

In [ ]:
ct = pd.read_csv(OUT + 'cleaned_transactions.csv')

daily = ct.groupby('transaction_date').apply(
    lambda x: pd.Series({
        'total_gmv_usd'      : round(x['amount_usd'].sum(), 2),
        'captured_gmv_usd'   : round(x.loc[x['status']=='captured','amount_usd'].sum(), 2),
        'total_transactions' : len(x),
        'success_count'      : (x['status']=='captured').sum(),
        'chargeback_count'   : (x['status']=='chargeback').sum(),
        'failed_count'       : (x['status']=='failed').sum(),
    })
).reset_index()
daily['success_rate_pct'] = round(daily['success_count']/daily['total_transactions']*100, 2)

display(daily)
daily.to_csv(OUT + 'daily_summary.csv', index=False)
print("Saved: daily_summary.csv")

### Payment Method Breakdown

In [ ]:
pm = ct.groupby('payment_method').apply(
    lambda x: pd.Series({
        'transaction_count'  : len(x),
        'total_gmv_usd'      : round(x['amount_usd'].sum(), 2),
        'captured_gmv_usd'   : round(x.loc[x['status']=='captured','amount_usd'].sum(), 2),
        'chargeback_count'   : (x['status']=='chargeback').sum(),
        'failed_count'       : (x['status']=='failed').sum(),
    })
).reset_index()

display(pm)
pm.to_csv(OUT + 'payment_method_breakdown.csv', index=False)
print("Saved: payment_method_breakdown.csv")

### Region Breakdown

In [ ]:
rb = ct.groupby('gateway_region').apply(
    lambda x: pd.Series({
        'transaction_count'  : len(x),
        'total_gmv_usd'      : round(x['amount_usd'].sum(), 2),
        'captured_gmv_usd'   : round(x.loc[x['status']=='captured','amount_usd'].sum(), 2),
        'avg_risk_score'     : round(x['risk_score'].mean(), 2),
        'chargeback_count'   : (x['status']=='chargeback').sum(),
        'failed_count'       : (x['status']=='failed').sum(),
        'high_value_count'   : x['high_value_flag'].sum(),
        'high_risk_count'    : x['high_risk_flag'].sum(),
    })
).reset_index()

display(rb)
rb.to_csv(OUT + 'region_breakdown.csv', index=False)
print("Saved: region_breakdown.csv")

### Merchant Performance Summary

In [ ]:
mp = ct.groupby(['merchant_id','merchant_name','merchant_category']).apply(
    lambda x: pd.Series({
        'total_transactions'  : len(x),
        'total_gmv_usd'       : round(x['amount_usd'].sum(), 2),
        'captured_gmv_usd'    : round(x.loc[x['status']=='captured','amount_usd'].sum(), 2),
        'avg_risk_score'      : round(x['risk_score'].mean(), 2),
        'chargeback_count'    : (x['status']=='chargeback').sum(),
        'failed_count'        : (x['status']=='failed').sum(),
        'high_value_count'    : x['high_value_flag'].sum(),
        'high_risk_count'     : x['high_risk_flag'].sum(),
    })
).reset_index()
mp['chargeback_ratio_pct'] = round(mp['chargeback_count']/mp['total_transactions']*100, 2)

display(mp)
mp.to_csv(OUT + 'merchant_performance_summary.csv', index=False)
print("Saved: merchant_performance_summary.csv")

---
## ✅ Pipeline Complete

All output files generated:
- `cleaned_transactions.csv`
- `merchant_risk_summary.csv`
- `missing_in_gateway.csv`
- `missing_in_ledger.csv`
- `amount_mismatches.csv`
- `status_mismatches.csv`
- `reconciliation_report.csv`
- `api_normalized.csv`
- `daily_summary.csv`
- `payment_method_breakdown.csv`
- `region_breakdown.csv`
- `merchant_performance_summary.csv`
- `summary_metrics.json`